短期记忆是 state 会话内部状态+checkpoint持久化机制+thread id会话作用域 三者的组合

1.2 基于内存级别的持久化器的实现 checkpointer config thread_id  checkpointer会自动管理历史，自动追加消息，更新message历史列表，然后把新的message历史列表发给大模型，

In [3]:

from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from win32netcon import RP_INIFILE

load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#2 创建内存级的存储
myCheckpointer=InMemorySaver()

#3 初始化1个agent实例
myAgent=create_agent(
    model=myModel,
    name="琳琳",
    #关键设置，让Agent具备了存储的能力
    checkpointer=myCheckpointer,
)
#4 创建1个config文件，保证同1个thread id是共享记忆的
myConfig={
    "configurable":{
        "thread_id":"Aureliano"
    }
}
print("---"*50,"第1轮对话")
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [ SystemMessage("你是1个专业的Ai助手"),
              HumanMessage("你好，我是Aureliano"),
            ],
    },
    config=myConfig
)

for msg in response["messages"]:
    msg.pretty_print()

print("---"*50,"第2轮对话")
response2=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
              HumanMessage("你知道我叫什么了吗"),
            ],
    },
    config=myConfig
)

for msg in response2["messages"]:
    msg.pretty_print()

------------------------------------------------------------------------------------------------------------------------------------------------------ 第1轮对话
================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

你好，我是Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？
------------------------------------------------------------------------------------------------------------------------------------------------------ 第2轮对话
================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

你好，我是Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message ===========

In [4]:
thread_id_1=myAgent.get_state(myConfig)
rprint(thread_id_1)

StateSnapshot(
    values={
        'messages': [
            SystemMessage(
                content='你是1个专业的Ai助手',
                additional_kwargs={},
                response_metadata={},
                id='6b5043e6-cd15-48a1-8791-93962d0918e4'
            ),
            HumanMessage(
                content='你好，我是Aureliano',
                additional_kwargs={},
                response_metadata={},
                id='7f5faf73-12b7-4c51-8ab7-03fb67e8b09f'
            ),
            AIMessage(
                content='你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 67,
                        'prompt_tokens': 66,
                        'total_tokens': 133,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 47,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 67
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': 66
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.8-max',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-78139298-f3ca-9084-b5e8-80a29c64b5d4',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                name='琳琳',
                id='lc_run--01a0b8cd-a3d3-7790-bac2-d461406cb8ad-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 66,
                    'output_tokens': 67,
                    'total_tokens': 133,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 47}
                }
            ),
            HumanMessage(
                content='你知道我叫什么了吗',
                additional_kwargs={},
                response_metadata={},
                id='060743e9-9cf3-4439-8d35-7e623099be11'
            ),
            AIMessage(
                content='当然知道，你叫 **Aureliano**！有什么我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 104,
                        'prompt_tokens': 101,
                        'total_tokens': 205,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 83,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 104
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': 101
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.8-max',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-65e386ad-73b4-9498-b2fc-969bfc6a9386',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                name='琳琳',
                id='lc_run--01a0b8cd-ae1b-7

In [5]:
print("---"*50,"第3轮对话")
response3=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
              HumanMessage("我刚才问了什么"),
            ],
    },
    config=myConfig
)

for msg in response3["messages"]:
    msg.pretty_print()

------------------------------------------------------------------------------------------------------------------------------------------------------ 第3轮对话
================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

你好，我是Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message =================================

你知道我叫什么了吗
================================== Ai Message ==================================
Name: 琳琳

当然知道，你叫 **Aureliano**！有什么我可以帮你的吗？
================================ Human Message =================================

我刚才问了什么
================================== Ai Message ==================================
Name: 琳琳

你刚才问的是：**“你知道我叫什么了吗”**。


更换1个config文件作为参照对象进行测试 结果显示不同的thread id不会共享记忆

In [6]:
print("---"*50,"第4轮对话")
newConfig={
    "configurable":{
        "thread_id":"Dived"
    }

}
response4=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
              HumanMessage("我叫什么"),
            ],
    },
    config=newConfig
)

for msg in response4["messages"]:
    msg.pretty_print()

------------------------------------------------------------------------------------------------------------------------------------------------------ 第4轮对话
================================ Human Message =================================

我叫什么
================================== Ai Message ==================================
Name: 琳琳

我还不知道你叫什么呢 😊  
你可以告诉我你的名字，或者想让我怎么称呼你。
